# GST Invoice Extractor — Production v11 (One-Run)

**Pipeline:** PDF/image → PP-StructureV3 OCR → LayoutLMv3 Model A + Model B → OCR/context reconciliation → GST/tax extraction → geometry-aware line-item reconstruction → validation → canonical JSON.

This version is designed for a fresh Google Colab runtime and fixes the dependency-bootstrap failure modes seen in v6/v7.

### Runtime guarantees
- Dependencies are installed **before any third-party import**.
- `PyMuPDF` provides `fitz`.
- `PaddleOCR` is explicitly installed and verified before invoice upload.
- PaddlePaddle/PaddleOCR versions are pinned.
- oneDNN is disabled before Paddle is imported.
- GPU is detected automatically; CPU remains a supported fallback.
- No Drive unmount, flush, deletion, or checkpoint deletion.
- `os.kill(os.getpid(), signal.SIGKILL)` is used only when a native package change genuinely requires a clean interpreter.
- No optional FormulaNet, table-recognition, seal, chart, orientation, or unwarping models are requested.
- PDF pages are processed one at a time.
- Temporary page images are deleted immediately.
- A page failure is reported clearly and does not masquerade as a successful extraction.

### v10 structural table fix
- The BILL/INVOICE table is detected from its header and terminated at its Total/Consumption Details boundary.
- Consumption Details rows are never emitted as invoice `line_items`.
- The four numeric bill columns are selected by fixed table order (GR QTY, BILL QTY, RATE, AMT), not by multiplication alone.
- Customer extraction prefers the invoice header and rejects material-table contamination.
- A second unlabeled `DATE` is retained only as unclassified evidence; it is never silently converted into `DUE_DATE`.


In [ ]:
# ============================================================
# CELL 0 — COMPLETE COLAB BOOTSTRAP
# MUST BE THE FIRST EXECUTABLE CELL
# ============================================================
# Important: DO NOT import paddle, paddleocr, fitz, transformers,
# cv2, PIL, etc. before this cell.
#
# We intentionally install packages with subprocess before importing
# native libraries. This avoids the previous "fitz missing" and
# "paddleocr missing" failures.
#
# If an already-loaded incompatible native package is detected, the
# notebook uses os.kill() exactly as requested. In a genuinely fresh
# Colab runtime, no kill is normally required.
# ============================================================

import os
import sys
import signal
import subprocess
import importlib.util
import importlib.metadata as importlib_metadata
import platform

# Prevent the CPU oneDNN/PIR path that caused the previous failure.
os.environ["FLAGS_use_mkldnn"] = "0"
os.environ["FLAGS_allocator_strategy"] = "auto_growth"
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

print("=" * 78)
print("GST INVOICE EXTRACTOR — v11 ONE-RUN BOOTSTRAP")
print("=" * 78)
print("Python:", platform.python_version())

# ------------------------------------------------------------
# Approved versions
# ------------------------------------------------------------
# PaddlePaddle 3.2.2 avoids the CPU oneDNN/PIR failure observed
# in the previous run. PaddleOCR 3.4.1 is kept aligned with it.
TARGETS = {
    "PyMuPDF": "PyMuPDF>=1.24,<2",
    "Pillow": "Pillow>=10,<13",
    "numpy": "numpy>=1.26,<3",
    "pandas": "pandas>=2,<3",
    "opencv-python-headless": "opencv-python-headless>=4.8,<5",
    "tqdm": "tqdm>=4.65,<5",
    "requests": "requests>=2.31,<3",
    "safetensors": "safetensors>=0.4,<1",
    "transformers": "transformers>=4.45,<5",
    "paddlepaddle": "paddlepaddle==3.2.2",
    "paddleocr": "paddleocr[doc-parser]==3.4.1",
}

IMPORTS = {
    "PyMuPDF": "fitz",
    "Pillow": "PIL",
    "numpy": "numpy",
    "pandas": "pandas",
    "opencv-python-headless": "cv2",
    "tqdm": "tqdm",
    "requests": "requests",
    "safetensors": "safetensors",
    "transformers": "transformers",
    "paddlepaddle": "paddle",
    "paddleocr": "paddleocr",
}

def installed_version(distribution):
    try:
        return importlib_metadata.version(distribution)
    except importlib_metadata.PackageNotFoundError:
        return None

def needs_install(distribution, spec):
    # Native packages need exact compatibility; simple version checks
    # are handled explicitly below.
    version = installed_version(distribution)
    if version is None:
        return True
    if distribution == "paddlepaddle":
        return version != "3.2.2"
    if distribution == "paddleocr":
        return version != "3.4.1"
    return False

missing_or_incompatible = [
    spec for distribution, spec in TARGETS.items()
    if needs_install(distribution, spec)
]

if missing_or_incompatible:
    print("\nInstalling / aligning:")
    for spec in missing_or_incompatible:
        print("  -", spec)

    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--no-cache-dir", "-q",
        *missing_or_incompatible
    ])

    importlib.invalidate_caches()

    # We have not imported the native packages in this cell, so a clean
    # process is normally unnecessary. If the current interpreter already
    # has native Paddle modules loaded from an earlier manual action,
    # force a clean runtime with os.kill().
    if any(name in sys.modules for name in ("paddle", "paddleocr", "paddlex")):
        print("\nNative Paddle modules were already loaded.")
        print("Restarting safely with os.kill()...")
        os.kill(os.getpid(), signal.SIGKILL)

print("\nDependency versions:")
for distribution in TARGETS:
    print(f"  {distribution:28s}: {installed_version(distribution) or 'MISSING'}")

# Final import-spec verification BEFORE proceeding.
still_missing = [
    module for module in IMPORTS.values()
    if importlib.util.find_spec(module) is None
]

if still_missing:
    raise RuntimeError(
        "Bootstrap failed; missing modules after installation: "
        + ", ".join(still_missing)
    )

print("\n[OK] All required modules are installed.")
print("=" * 78)


In [ ]:
# ============================================================
# CELL 1 — IMPORTS, DRIVE, PATHS, HARD ENVIRONMENT VALIDATION
# ============================================================

from google.colab import drive, files

drive.mount("/content/drive", force_remount=False)

import os
import re
import io
import json
import math
import time
import shutil
import datetime as dt
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import fitz
from PIL import Image

from transformers import (
    LayoutLMv3Processor,
    LayoutLMv3ForTokenClassification,
)

import paddle
import paddleocr
from paddleocr import PPStructureV3

PROJECT_DIR = Path("/content/drive/MyDrive/Enterprise_Document_AI")
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"

MODEL_A_DIR = CHECKPOINT_DIR / "layoutlmv3_field_extractor"
MODEL_B_DIR = CHECKPOINT_DIR / "layoutlmv3_gst_extractor"

OUTPUT_DIR = PROJECT_DIR / "output" / "gst_invoice_extractor_v10"
TEMP_DIR = OUTPUT_DIR / "_temp"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXTENSIONS = {
    ".pdf", ".jpg", ".jpeg", ".png", ".webp", ".tif", ".tiff"
}

CANONICAL_FIELDS = [
    "VENDOR_NAME",
    "CUSTOMER_NAME",
    "ADDRESS",
    "INVOICE_NUMBER",
    "INVOICE_DATE",
    "DUE_DATE",
    "REF_NO",
    "CENTRAL_GST",
    "STATE_GST",
    "SUBTOTAL",
    "TAX",
    "TOTAL_AMOUNT",
]

MONEY_FIELDS = {
    "CENTRAL_GST",
    "STATE_GST",
    "SUBTOTAL",
    "TAX",
    "TOTAL_AMOUNT",
}

device = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 78)
print("ENVIRONMENT VALIDATION")
print("=" * 78)
print("Python       :", platform.python_version() if "platform" in globals() else sys.version.split()[0])
print("PyTorch      :", torch.__version__)
print("Transformers :", __import__("transformers").__version__)
print("PyMuPDF      :", fitz.__doc__.splitlines()[0] if fitz.__doc__ else "installed")
print("Paddle       :", paddle.__version__)
print("PaddleOCR    :", getattr(paddleocr, "__version__", "installed"))
print("Device       :", device)

if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))
    print("VRAM GB      :", round(
        torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2
    ))
else:
    print("GPU          : NOT AVAILABLE")

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"Project directory not found: {PROJECT_DIR}"
    )

for model_dir, name in [
    (MODEL_A_DIR, "Model A"),
    (MODEL_B_DIR, "Model B"),
]:
    if not model_dir.exists():
        raise FileNotFoundError(
            f"{name} directory not found: {model_dir}"
        )

print("Project      :", PROJECT_DIR)
print("Model A dir  :", MODEL_A_DIR)
print("Model B dir  :", MODEL_B_DIR)
print("Output dir   :", OUTPUT_DIR)
print("=" * 78)


In [ ]:
# ============================================================
# CELL 2 — SAFE CHECKPOINT DISCOVERY + LAYOUTLM LOADING
# ============================================================

def valid_checkpoint(path):
    path = Path(path)
    if not path.is_dir():
        return False
    return (
        (path / "config.json").exists()
        and (
            (path / "model.safetensors").exists()
            or (path / "pytorch_model.bin").exists()
        )
    )

def find_model_source(model_dir):
    model_dir = Path(model_dir)

    if valid_checkpoint(model_dir):
        return model_dir

    candidates = []

    # Search both direct Trainer checkpoints and training_runs/checkpoints.
    search_roots = [model_dir, model_dir / "training_runs"]

    for root in search_roots:
        if not root.exists():
            continue
        for p in root.glob("checkpoint-*"):
            if valid_checkpoint(p):
                m = re.search(r"checkpoint-(\d+)$", p.name)
                step = int(m.group(1)) if m else -1
                candidates.append((step, p))

    if not candidates:
        raise FileNotFoundError(
            f"No valid LayoutLMv3 checkpoint under {model_dir}"
        )

    candidates.sort(key=lambda x: x[0], reverse=True)
    return candidates[0][1]

def load_layoutlm(model_dir, model_name):
    source = find_model_source(model_dir)

    print(f"\nLoading {model_name}")
    print("Source:", source)

    processor = LayoutLMv3Processor.from_pretrained(
        str(source),
        apply_ocr=False,
    )

    model = LayoutLMv3ForTokenClassification.from_pretrained(
        str(source)
    )

    model.to(device)
    model.eval()

    id2label = {
        int(k): str(v)
        for k, v in model.config.id2label.items()
    }

    if not id2label:
        raise ValueError(f"{model_name} has no id2label configuration.")

    print("Labels:", id2label)
    print("Device:", device)

    return model, processor, id2label, source

model_a, model_a_processor, model_a_id2label, model_a_source = load_layoutlm(
    MODEL_A_DIR, "MODEL A — universal invoice extractor"
)

model_b, model_b_processor, model_b_id2label, model_b_source = load_layoutlm(
    MODEL_B_DIR, "MODEL B — GST-specific extractor"
)

print("\n[OK] Both LayoutLMv3 models loaded.")


In [ ]:
# ============================================================
# CELL 3 — ROBUST PP-STRUCTUREV3 OCR ADAPTER
# ============================================================
# FIX v10:
# PP-StructureV3 in PaddleOCR 3.x can expose OCR results as:
#   res.rec_texts / res.rec_scores / res.rec_boxes
# or inside res.json / res["res"].
#
# The previous v8 recursive walker could miss those arrays and therefore
# falsely report "zero OCR records" even when OCR had actually succeeded.
#
# This cell:
# 1) extracts the native PaddleOCR 3.x arrays directly;
# 2) handles json properties that are dicts, strings, or callables;
# 3) supports legacy nested OCR structures;
# 4) falls back to PaddleOCR's direct OCR pipeline if PP-StructureV3
#    exposes no usable OCR records.
# ============================================================

def bbox4(box):
    """Convert polygon/box-like input to [x1,y1,x2,y2]."""
    if box is None:
        raise ValueError("Empty OCR box")

    arr = np.asarray(box, dtype=float)

    if arr.ndim == 1 and arr.size >= 4:
        # x1,y1,x2,y2
        return [
            float(arr[0]), float(arr[1]),
            float(arr[2]), float(arr[3])
        ]

    if arr.ndim == 2 and arr.shape[1] >= 2:
        xs = arr[:, 0]
        ys = arr[:, 1]
        return [
            float(xs.min()), float(ys.min()),
            float(xs.max()), float(ys.max())
        ]

    flat = arr.reshape(-1)
    if flat.size >= 4:
        return [
            float(flat[0]), float(flat[1]),
            float(flat[2]), float(flat[3])
        ]

    raise ValueError(f"Unsupported OCR box: {box}")


def _materialize_json(obj):
    """Return an OCR result's JSON payload when available."""
    if obj is None:
        return None

    try:
        value = getattr(obj, "json")
    except Exception:
        return obj

    try:
        value = value() if callable(value) else value
    except Exception:
        return obj

    if isinstance(value, str):
        try:
            return json.loads(value)
        except Exception:
            return obj

    return value


def _append_parallel_ocr_arrays(payload, records):
    """
    Extract PaddleOCR 3.x arrays:
      rec_texts + rec_boxes/rec_polys + rec_scores
    Also handles payload['res'].
    """
    if not isinstance(payload, dict):
        return 0

    added = 0

    candidates = [payload]
    nested = payload.get("res")
    if isinstance(nested, dict):
        candidates.insert(0, nested)

    for obj in candidates:
        texts = obj.get("rec_texts")
        if texts is None:
            texts = obj.get("texts")

        scores = obj.get("rec_scores")
        if scores is None:
            scores = obj.get("scores")

        boxes = obj.get("rec_boxes")
        if boxes is None:
            boxes = obj.get("rec_polys")
        if boxes is None:
            boxes = obj.get("dt_polys")
        if boxes is None:
            boxes = obj.get("boxes")
        if boxes is None:
            boxes = obj.get("polys")

        if texts is None or boxes is None:
            continue

        texts = list(texts)
        boxes = list(boxes)

        if scores is None:
            scores = [1.0] * len(texts)
        else:
            scores = list(scores)

        n = min(len(texts), len(boxes))

        for i in range(n):
            text = str(texts[i]).strip()
            if not text:
                continue

            try:
                box = bbox4(boxes[i])
            except Exception:
                continue

            try:
                score = float(scores[i]) if i < len(scores) else 1.0
            except Exception:
                score = 1.0

            records.append({
                "word": text,
                "box": box,
                "confidence": score,
            })
            added += 1

        if added:
            # Do not parse the same parallel arrays twice.
            break

    return added


def _walk_paddle(obj, records, depth=0):
    """
    Recursive compatibility extractor for older/nested PaddleOCR outputs.
    """
    if obj is None or depth > 20:
        return

    # First try native JSON representation.
    materialized = _materialize_json(obj)
    if materialized is not obj:
        _walk_paddle(materialized, records, depth + 1)
        return

    if isinstance(obj, dict):
        # Native PaddleOCR 3.x arrays.
        _append_parallel_ocr_arrays(obj, records)

        text = obj.get("text")
        if text is None:
            text = obj.get("transcription")

        box = obj.get("bbox")
        if box is None:
            box = obj.get("box")
        if box is None:
            box = obj.get("points")
        if box is None:
            box = obj.get("coordinate")

        score = obj.get("confidence")
        if score is None:
            score = obj.get("score")
        if score is None:
            score = obj.get("rec_score")

        if text is not None and box is not None:
            try:
                records.append({
                    "word": str(text).strip(),
                    "box": bbox4(box),
                    "confidence": float(score) if score is not None else 1.0,
                })
            except Exception:
                pass

        for value in obj.values():
            _walk_paddle(value, records, depth + 1)

    elif isinstance(obj, (list, tuple)):
        # Standard OCR tuple:
        # [polygon, (text, score)]
        if (
            len(obj) == 2
            and isinstance(obj[1], (list, tuple))
            and len(obj[1]) >= 1
            and isinstance(obj[1][0], str)
        ):
            try:
                records.append({
                    "word": str(obj[1][0]).strip(),
                    "box": bbox4(obj[0]),
                    "confidence": (
                        float(obj[1][1])
                        if len(obj[1]) > 1 else 1.0
                    ),
                })
                return
            except Exception:
                pass

        for value in obj:
            _walk_paddle(value, records, depth + 1)


def dedupe_ocr_records(records):
    output = []
    seen = set()

    for record in records:
        word = str(record["word"]).strip()
        if not word:
            continue

        try:
            box = bbox4(record["box"])
        except Exception:
            continue

        key = (
            word,
            tuple(round(float(v), 1) for v in box)
        )

        if key in seen:
            continue

        seen.add(key)
        output.append({
            "word": word,
            "box": box,
            "confidence": float(record.get("confidence", 1.0)),
        })

    output.sort(
        key=lambda r: (
            (r["box"][1] + r["box"][3]) / 2,
            r["box"][0],
        )
    )
    return output


# ------------------------------------------------------------
# OCR engine
# ------------------------------------------------------------
PP_STRUCTURE = None
DIRECT_OCR = None

PADDLE_DEVICE = (
    "gpu:0"
    if (
        paddle.device.is_compiled_with_cuda()
        and torch.cuda.is_available()
    )
    else "cpu"
)


def get_pp_structure():
    global PP_STRUCTURE

    if PP_STRUCTURE is not None:
        return PP_STRUCTURE

    os.environ["FLAGS_use_mkldnn"] = "0"
    os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

    kwargs = {
        "use_doc_orientation_classify": False,
        "use_doc_unwarping": False,
        "use_textline_orientation": False,
        "use_seal_recognition": False,
        "use_table_recognition": False,
        "use_formula_recognition": False,
        "use_chart_recognition": False,
        "use_region_detection": False,
        "device": PADDLE_DEVICE,
    }

    try:
        PP_STRUCTURE = PPStructureV3(**kwargs)
    except TypeError:
        # Constructor compatibility for builds exposing fewer arguments.
        minimal = {
            "use_doc_orientation_classify": False,
            "use_doc_unwarping": False,
            "use_textline_orientation": False,
            "use_table_recognition": False,
            "use_formula_recognition": False,
            "device": PADDLE_DEVICE,
        }
        PP_STRUCTURE = PPStructureV3(**minimal)

    print("[OK] PP-StructureV3 initialized on", PADDLE_DEVICE)
    return PP_STRUCTURE


def _get_direct_ocr():
    global DIRECT_OCR

    if DIRECT_OCR is not None:
        return DIRECT_OCR

    from paddleocr import PaddleOCR

    kwargs = {
        "lang": "en",
        "use_doc_orientation_classify": False,
        "use_doc_unwarping": False,
        "use_textline_orientation": False,
        "device": PADDLE_DEVICE,
    }

    try:
        DIRECT_OCR = PaddleOCR(**kwargs)
    except TypeError:
        minimal = {
            "lang": "en",
            "use_doc_orientation_classify": False,
            "use_doc_unwarping": False,
            "use_textline_orientation": False,
            "device": PADDLE_DEVICE,
        }
        DIRECT_OCR = PaddleOCR(**minimal)

    print("[OK] Direct PaddleOCR fallback initialized on", PADDLE_DEVICE)
    return DIRECT_OCR


def _run_and_extract(engine, image_path):
    results = engine.predict(str(image_path))
    records = []

    for result in results:
        # 1. Native object representation.
        payload = _materialize_json(result)

        # 2. Direct parallel-array extraction.
        if isinstance(payload, dict):
            _append_parallel_ocr_arrays(payload, records)

        # 3. Recursive compatibility extraction.
        _walk_paddle(payload, records)

        # 4. Some result objects expose useful attributes directly.
        if not records:
            try:
                texts = getattr(result, "rec_texts", None)
                boxes = getattr(result, "rec_boxes", None)
                if boxes is None:
                    boxes = getattr(result, "dt_polys", None)
                scores = getattr(result, "rec_scores", None)

                if texts is not None and boxes is not None:
                    texts = list(texts)
                    boxes = list(boxes)
                    scores = (
                        list(scores)
                        if scores is not None
                        else [1.0] * len(texts)
                    )

                    for i in range(min(len(texts), len(boxes))):
                        text = str(texts[i]).strip()
                        if not text:
                            continue

                        records.append({
                            "word": text,
                            "box": bbox4(boxes[i]),
                            "confidence": (
                                float(scores[i])
                                if i < len(scores) else 1.0
                            ),
                        })
            except Exception:
                pass

    return dedupe_ocr_records(records)


def ocr_image_pp_structure(image_path):
    # Primary path.
    engine = get_pp_structure()
    records = _run_and_extract(engine, image_path)

    if records:
        words = [r["word"] for r in records]
        boxes = [r["box"] for r in records]
        approx = [r["confidence"] < 0.5 for r in records]

        print(f"OCR records: {len(records)}")
        return words, boxes, approx, "pp_structure_v3"

    # Fallback path:
    # If PP-StructureV3 returns a result object but no usable OCR arrays,
    # run the lightweight OCR-only PaddleOCR pipeline instead of stopping.
    print("PP-StructureV3 exposed no OCR records; using direct PaddleOCR fallback.")

    direct = _get_direct_ocr()
    records = _run_and_extract(direct, image_path)

    if not records:
        raise RuntimeError(
            "Both PP-StructureV3 and direct PaddleOCR returned zero "
            f"OCR records for {image_path}"
        )

    words = [r["word"] for r in records]
    boxes = [r["box"] for r in records]
    approx = [r["confidence"] < 0.5 for r in records]

    print(f"OCR records: {len(records)}")
    return words, boxes, approx, "paddleocr_direct_fallback"


# Eager initialization BEFORE upload.
get_pp_structure()


In [ ]:
# ============================================================
# CELL 4 — GEOMETRY / ROW RECONSTRUCTION
# ============================================================

def y_center(box):
    return (box[1] + box[3]) / 2.0

def x_center(box):
    return (box[0] + box[2]) / 2.0

def row_height(box):
    return max(1.0, box[3] - box[1])

def candidate_rows(words, boxes):
    if len(words) != len(boxes):
        raise ValueError("OCR words/boxes length mismatch")

    indexed = sorted(
        range(len(words)),
        key=lambda i: (y_center(boxes[i]), boxes[i][0])
    )

    rows = []

    for idx in indexed:
        yc = y_center(boxes[idx])
        h = row_height(boxes[idx])

        best = None
        best_delta = None

        for row in rows:
            tolerance = max(
                8.0,
                0.55 * max(h, row["median_height"])
            )
            delta = abs(yc - row["y_center"])

            if delta <= tolerance:
                if best_delta is None or delta < best_delta:
                    best = row
                    best_delta = delta

        if best is None:
            rows.append({
                "indices": [idx],
                "y_center": yc,
                "median_height": h,
            })
        else:
            best["indices"].append(idx)

            ys = [
                y_center(boxes[i])
                for i in best["indices"]
            ]
            hs = [
                row_height(boxes[i])
                for i in best["indices"]
            ]

            best["y_center"] = float(np.median(ys))
            best["median_height"] = float(np.median(hs))

    for row in rows:
        row["indices"].sort(key=lambda i: boxes[i][0])
        row["text"] = " ".join(words[i] for i in row["indices"])
        row["x1"] = min(boxes[i][0] for i in row["indices"])
        row["x2"] = max(boxes[i][2] for i in row["indices"])

    rows.sort(key=lambda r: r["y_center"])
    return rows

def clean_text(value):
    return re.sub(r"\s+", " ", str(value or "")).strip()

def ocr_numeric_normalize(value):
    s = str(value).strip()
    s = s.replace("₹", "")
    s = re.sub(r"\bRs\.?\b", "", s, flags=re.I)
    s = s.replace("O", "0").replace("o", "0")
    s = s.replace("I", "1").replace("l", "1")
    s = s.replace("|", "1")
    return s.replace(" ", "")

def normalize_money(value):
    if value is None:
        return None

    s = ocr_numeric_normalize(value)
    s = re.sub(r"[^0-9.\-]", "", s)

    if not s or s in {"-", "."}:
        return None

    if s.count("-") > 1 or ("-" in s and not s.startswith("-")):
        return None

    try:
        n = float(s)
    except Exception:
        return None

    if not math.isfinite(n):
        return None

    return f"{n:.2f}"

def money_float(value):
    v = normalize_money(value)
    return float(v) if v is not None else None

def looks_like_reference_number(value):
    v = normalize_money(value)
    if v is None:
        return False

    digits = re.sub(r"\D", "", v.split(".")[0])

    # Invoice/reference/document IDs commonly have 7–12+ digits and
    # should never become a line-item amount merely because a regex saw them.
    if len(digits) >= 7:
        return True

    return False

def is_percent_token(text):
    t = str(text).strip()
    if "%" in t:
        return True

    v = money_float(t)
    return v is not None and 0 <= v <= 50


In [ ]:
# ============================================================
# CELL 5 — GSTIN / DATE / INVOICE NUMBER / FINANCIAL EXTRACTION
# ============================================================

GSTIN_RE = re.compile(
    r"\b[0-9]{2}[A-Z]{5}[0-9]{4}[A-Z][A-Z0-9]Z[A-Z0-9]\b",
    re.I,
)

DATE_RE = re.compile(
    r"\b(?:"
    r"\d{1,2}[./-]\d{1,2}[./-]\d{2,4}"
    r"|"
    r"\d{4}[./-]\d{1,2}[./-]\d{1,2}"
    r")\b"
)

GST_STATE_CODES = {f"{i:02d}" for i in range(1, 38)}

def normalize_date(value):
    value = clean_text(value)
    if not value:
        return None

    # OCR may return a label together with the date, e.g.
    # "DATE : 07.04.2026". Extract only the actual date token.
    found = DATE_RE.search(value)
    if found:
        value = found.group(0)
    value = value.replace(" ", "")

    formats = [
        "%d.%m.%Y", "%d/%m/%Y", "%d-%m-%Y",
        "%d.%m.%y", "%d/%m/%y", "%d-%m-%y",
        "%Y-%m-%d", "%Y/%m/%d", "%Y.%m.%d",
    ]

    for fmt in formats:
        try:
            parsed = dt.datetime.strptime(value, fmt)
            if parsed.year < 2000:
                parsed = parsed.replace(year=parsed.year + 2000)
            return parsed.strftime("%Y-%m-%d")
        except ValueError:
            continue

    return None

def extract_gstins(words, boxes):
    rows = candidate_rows(words, boxes)
    found = []

    for row in rows:
        idxs = row["indices"]

        # Individual and up to 4-token same-row concatenation.
        for start in range(len(idxs)):
            for width in range(1, min(4, len(idxs) - start) + 1):
                token_indices = idxs[start:start + width]
                candidate = "".join(
                    re.sub(r"[^A-Za-z0-9]", "", words[i])
                    for i in token_indices
                ).upper()

                # OCR confusions only for characters whose expected position
                # is unambiguous.
                candidate = (
                    candidate
                    .replace(" ", "")
                    .replace("—", "")
                )

                for match in GSTIN_RE.finditer(candidate):
                    value = match.group(0).upper()
                    if value[:2] in GST_STATE_CODES:
                        found.append({
                            "value": value,
                            "raw": " ".join(words[i] for i in token_indices),
                            "confidence": 1.0,
                        })

    # Individual tokens too.
    for word in words:
        candidate = re.sub(r"[^A-Za-z0-9]", "", str(word)).upper()
        for match in GSTIN_RE.finditer(candidate):
            value = match.group(0).upper()
            if value[:2] in GST_STATE_CODES:
                found.append({
                    "value": value,
                    "raw": word,
                    "confidence": 1.0,
                })

    # Dedupe.
    unique = {}
    for item in found:
        unique[item["value"]] = item

    return list(unique.values())

def extract_labeled_value(rows, labels, exclude_labels=()):
    labels = [x.upper() for x in labels]
    excludes = [x.upper() for x in exclude_labels]

    candidates = []

    for row in rows:
        t = row["text"].upper()

        if any(e in t for e in excludes):
            continue

        matched = None
        for label in labels:
            if label in t:
                matched = label
                break

        if not matched:
            continue

        after = t.split(matched, 1)[1]
        dates = DATE_RE.findall(after)

        if dates:
            return {
                "value": dates[0],
                "raw_row": row["text"],
                "label": matched,
            }

    return None

def extract_invoice_number(rows):
    labels = [
        "BILL NO", "BILL NUMBER", "INVOICE NO",
        "INVOICE NUMBER", "INVOICE #", "INV NO",
    ]

    bad = [
        "REF NO", "REFERENCE NO", "FORM NO",
    ]

    for row in rows:
        t = row["text"]
        u = t.upper()

        if any(b in u for b in bad):
            continue

        for label in labels:
            pos = u.find(label)
            if pos < 0:
                continue

            tail = t[pos + len(label):]
            tail = re.sub(r"^[\s:#\-./]+", " ", tail).strip()

            m = re.search(r"\b[A-Za-z0-9][A-Za-z0-9./_-]{2,30}\b", tail)
            if m:
                value = m.group(0).strip(" .,:;")
                if not DATE_RE.fullmatch(value):
                    return {
                        "value": value,
                        "raw": tail,
                        "box": [row["x1"], row["y_center"], row["x2"], row["y_center"]],
                    }

    return None

MONEY_ROW_RULES = {
    "CENTRAL_GST": [
        ("CENTRAL GST",), ("CGST",),
    ],
    "STATE_GST": [
        ("STATE GST",), ("SGST",),
    ],
    "SUBTOTAL": [
        ("BASIC TOTAL",), ("SUB TOTAL",), ("SUBTOTAL",),
        ("TAXABLE VALUE",),
    ],
    "TOTAL_AMOUNT": [
        ("BILL AMOUNT",), ("GRAND TOTAL",),
        ("INVOICE TOTAL",), ("TOTAL AMOUNT",),
        ("TOTAL VALUE",), ("NET PAYABLE",),
    ],
}

def numeric_candidates(row):
    values = []

    for idx in row["indices"]:
        token = words_global[idx]
        normalized = normalize_money(token)

        if normalized is None:
            continue

        if is_percent_token(token):
            continue

        if looks_like_reference_number(normalized):
            continue

        values.append((idx, normalized))

    return values

def extract_money_fields(words, boxes):
    global words_global
    words_global = words

    rows = candidate_rows(words, boxes)
    output = {}
    diagnostics = defaultdict(list)

    for field, rules in MONEY_ROW_RULES.items():
        scored = []

        for row in rows:
            upper = row["text"].upper()

            for rule_group in rules:
                matched = next(
                    (label for label in rule_group if label in upper),
                    None
                )
                if not matched:
                    continue

                nums = numeric_candidates(row)

                # Financial rows can contain multiple numbers. Prefer the
                # right-most monetary value; exclude rates and IDs.
                for idx, value in nums:
                    x = boxes[idx][0]
                    scored.append({
                        "value": value,
                        "raw": words[idx],
                        "keyword": matched,
                        "row_text": row["text"],
                        "x": x,
                        "y": row["y_center"],
                    })

        if not scored:
            continue

        # For summary rows, the right-most valid amount is normally the
        # actual amount column. Explicit label beats generic proximity.
        scored.sort(key=lambda c: (c["y"], c["x"]))

        selected = scored[-1]
        output[field] = selected
        diagnostics[field] = scored

    # TAX = CGST + SGST when both are confidently extracted.
    cgst = money_float(output.get("CENTRAL_GST", {}).get("value"))
    sgst = money_float(output.get("STATE_GST", {}).get("value"))

    if cgst is not None and sgst is not None:
        output["TAX"] = {
            "value": f"{cgst + sgst:.2f}",
            "raw": f"{cgst:.2f} + {sgst:.2f}",
            "keyword": "CGST+SGST",
            "row_text": "derived",
        }

    return output, diagnostics

print("[OK] Financial and GST extraction helpers ready.")


In [ ]:
# ============================================================
# CELL 6 — LAYOUTLMv3 LONG-DOCUMENT INFERENCE
# ============================================================

def clamp_box(box, width, height):
    x1, y1, x2, y2 = box
    return [
        max(0, min(width, x1)),
        max(0, min(height, y1)),
        max(0, min(width, x2)),
        max(0, min(height, y2)),
    ]

def normalize_box_1000(box, width, height):
    x1, y1, x2, y2 = clamp_box(box, width, height)
    return [
        int(round(1000 * x1 / max(width, 1))),
        int(round(1000 * y1 / max(height, 1))),
        int(round(1000 * x2 / max(width, 1))),
        int(round(1000 * y2 / max(height, 1))),
    ]

def make_chunks(n, chunk_size=180, overlap=30):
    if n <= chunk_size:
        return [(0, n)]

    chunks = []
    start = 0

    while start < n:
        end = min(n, start + chunk_size)
        chunks.append((start, end))

        if end == n:
            break

        start = end - overlap

    return chunks

@torch.inference_mode()
def run_token_classifier(
    image,
    words,
    boxes,
    model,
    processor,
    id2label,
):
    width, height = image.size
    predictions = []

    for start, end in make_chunks(len(words)):
        chunk_words = words[start:end]
        chunk_boxes = boxes[start:end]

        normalized_boxes = [
            normalize_box_1000(b, width, height)
            for b in chunk_boxes
        ]

        encoded = processor(
            image,
            chunk_words,
            boxes=normalized_boxes,
            truncation=True,
            padding="max_length",
            max_length=512,
            return_tensors="pt",
        )

        encoded = {
            k: v.to(device)
            for k, v in encoded.items()
            if hasattr(v, "to")
        }

        outputs = model(**encoded)
        probs = torch.softmax(outputs.logits, dim=-1)
        conf, pred_ids = probs.max(dim=-1)

        # LayoutLM tokenization can split words. word_ids maps each model
        # token back to the original OCR word.
        word_ids = encoded.get("input_ids")
        try:
            mapping = processor.tokenizer(
                chunk_words,
                boxes=normalized_boxes,
                truncation=True,
                padding="max_length",
                max_length=512,
            ).word_ids()
        except Exception:
            mapping = [None] * pred_ids.shape[1]

        seen_word = set()

        for token_pos, word_id in enumerate(mapping):
            if word_id is None or word_id in seen_word:
                continue

            if word_id >= len(chunk_words):
                continue

            seen_word.add(word_id)

            global_idx = start + word_id
            label = id2label.get(
                int(pred_ids[0, token_pos].item()),
                "O"
            )

            predictions.append({
                "word_index": global_idx,
                "word": words[global_idx],
                "box": boxes[global_idx],
                "label": label,
                "confidence": float(conf[0, token_pos].item()),
            })

    return predictions

def canonical_field(label):
    if not label:
        return None

    value = str(label).upper()
    value = re.sub(r"^[BIOES]-", "", value)
    value = value.replace(" ", "_").replace("-", "_")

    aliases = {
        "VENDOR": "VENDOR_NAME",
        "VENDOR_NAME": "VENDOR_NAME",
        "SUPPLIER": "VENDOR_NAME",
        "SUPPLIER_NAME": "VENDOR_NAME",

        "CUSTOMER": "CUSTOMER_NAME",
        "CUSTOMER_NAME": "CUSTOMER_NAME",
        "BUYER": "CUSTOMER_NAME",

        "ADDRESS": "ADDRESS",
        "BILLING_ADDRESS": "ADDRESS",

        "INVOICE_NO": "INVOICE_NUMBER",
        "INVOICE_NUMBER": "INVOICE_NUMBER",
        "BILL_NO": "INVOICE_NUMBER",

        "INVOICE_DATE": "INVOICE_DATE",
        "DATE": "INVOICE_DATE",
        "DUE_DATE": "DUE_DATE",

        "REF_NO": "REF_NO",
        "REFERENCE_NO": "REF_NO",

        "CGST": "CENTRAL_GST",
        "CENTRAL_GST": "CENTRAL_GST",
        "SGST": "STATE_GST",
        "STATE_GST": "STATE_GST",

        "SUBTOTAL": "SUBTOTAL",
        "BASIC_TOTAL": "SUBTOTAL",
        "TAX": "TAX",
        "TOTAL": "TOTAL_AMOUNT",
        "TOTAL_AMOUNT": "TOTAL_AMOUNT",
        "GRAND_TOTAL": "TOTAL_AMOUNT",
    }

    return aliases.get(value)

def build_spans(predictions):
    spans = []
    current = None

    for pred in predictions:
        label = canonical_field(pred["label"])
        if label is None:
            if current:
                spans.append(current)
                current = None
            continue

        if current is None or current["label"] != label:
            if current:
                spans.append(current)

            current = {
                "label": label,
                "word_indices": [pred["word_index"]],
                "words": [pred["word"]],
                "boxes": [pred["box"]],
                "confidences": [pred["confidence"]],
            }
        else:
            current["word_indices"].append(pred["word_index"])
            current["words"].append(pred["word"])
            current["boxes"].append(pred["box"])
            current["confidences"].append(pred["confidence"])

    if current:
        spans.append(current)

    for span in spans:
        span["value"] = clean_text(" ".join(span["words"]))
        span["confidence"] = float(np.mean(span["confidences"]))
        span["box"] = [
            min(b[0] for b in span["boxes"]),
            min(b[1] for b in span["boxes"]),
            max(b[2] for b in span["boxes"]),
            max(b[3] for b in span["boxes"]),
        ]

    return spans


In [ ]:
# ============================================================
# CELL 7 — SEMANTIC FIELD + INVOICE-TABLE RECONSTRUCTION v10
# ============================================================
# Critical fix for invoices containing multiple tables.
#
# This invoice has:
#   1) BILL/INVOICE item table: PO NO / MATERIAL DESCRIPTION /
#      GR QTY / BILL QTY / RATE / AMT INR
#   2) Consumption Details table: Material Code / Material Desc /
#      Do No. / Do Date / Issue Qty / Receive Qty
#
# ONLY table (1) is allowed to populate line_items.
# ============================================================

SUMMARY_WORDS = {
    "SUBTOTAL", "BASIC TOTAL", "TAXABLE VALUE", "CGST", "CENTRAL GST",
    "SGST", "STATE GST", "IGST", "TAX", "GRAND TOTAL", "BILL AMOUNT",
    "TOTAL AMOUNT", "TOTAL VALUE", "NET PAYABLE", "ROUND OFF",
    "AMOUNT IN WORDS", "TERMS", "CONDITION", "REMARKS", "NOTE",
}

HEADER_WORDS = {
    "DESCRIPTION", "ITEM", "PRODUCT", "QTY", "QUANTITY", "RATE",
    "UNIT PRICE", "PRICE", "AMOUNT", "TOTAL", "HSN", "SAC",
}

CONSUMPTION_MARKERS = (
    "CONSUMPTION DETAILS",
    "MATERIAL CODE",
    "MATERIAL DESC",
    "DO NO",
    "DO DATE",
    "ISSUE QTY",
    "RECEIVE QTY",
    "DEBIT/CREDIT",
)

BILL_HEADER_MARKERS = (
    "PO NO",
    "MATERIAL DESCRIPTION",
    "GR QTY",
    "BILL QTY",
    "RATE",
    "AMT INR",
)


def _norm_upper(text):
    return re.sub(r"\s+", " ", str(text or "")).strip().upper()


def is_summary_row(text):
    t = _norm_upper(text)
    return any(k in t for k in SUMMARY_WORDS)


def is_header_row(text):
    t = _norm_upper(text)
    hits = sum(k in t for k in HEADER_WORDS)
    return hits >= 2


def is_consumption_row(text):
    t = _norm_upper(text)
    return any(k in t for k in CONSUMPTION_MARKERS)


def find_bill_table_bounds(rows):
    """
    Locate the actual invoice billing table, not the Consumption Details table.
    Returns (header_index, end_index) or (None, None).
    """
    header_idx = None

    for i, row in enumerate(rows):
        t = _norm_upper(row["text"])
        score = sum(marker in t for marker in BILL_HEADER_MARKERS)
        if score >= 3 and ("PO" in t or "MATERIAL" in t) and "QTY" in t:
            header_idx = i
            break

    if header_idx is None:
        return None, None

    end_idx = len(rows)

    for i in range(header_idx + 1, len(rows)):
        t = _norm_upper(rows[i]["text"])
        if "CONSUMPTION DETAILS" in t:
            end_idx = i
            break
        if re.search(r"\bTOTAL\b", t) and not any(
            x in t for x in ("GST", "VALUE", "AMOUNT IN WORDS")
        ):
            # The first Total immediately after invoice rows closes the table.
            end_idx = i
            break

    return header_idx, end_idx


def row_numbers(row, words, boxes):
    result = []
    for idx in row["indices"]:
        token = str(words[idx]).strip()
        if not token or "%" in token:
            continue

        # Dates are not invoice numeric columns.
        if DATE_RE.fullmatch(token):
            continue

        value = normalize_money(token)
        if value is None:
            continue

        # Long pure IDs are PO/DO/reference/material identifiers.
        if looks_like_reference_number(value):
            continue

        result.append({
            "index": idx,
            "value": value,
            "x": x_center(boxes[idx]),
            "raw": token,
        })

    return sorted(result, key=lambda n: n["x"])


def bill_row_numeric_columns(row, words, boxes):
    """
    BILL table schema is visually:
        PO | description | GR QTY | BILL QTY | RATE | AMT

    Therefore after rejecting the PO/reference number, the last four numeric
    tokens are deterministic: GR_QTY, BILL_QTY, RATE, AMT.
    """
    nums = row_numbers(row, words, boxes)
    if len(nums) < 4:
        return None

    # Use the final four numeric columns. This avoids choosing a combination
    # merely because multiplication happens to work.
    tail = nums[-4:]
    gr_qty, bill_qty, rate, amount = tail

    q1 = money_float(gr_qty["value"])
    q2 = money_float(bill_qty["value"])
    r = money_float(rate["value"])
    a = money_float(amount["value"])

    if any(v is None or v <= 0 for v in (q1, q2, r, a)):
        return None

    # Bill quantity should match GR quantity on this invoice family.
    qty_error = abs(q1 - q2) / max(abs(q2), 1.0)

    # Arithmetic is a validation signal, not the mechanism that selects columns.
    arithmetic_error = abs(q2 * r - a) / max(a, 1.0)

    # Invoice line amounts are allowed normal rounding only.
    if arithmetic_error > 0.02:
        return None

    return {
        "quantity": bill_qty["value"],
        "unit_price": rate["value"],
        "amount": amount["value"],
        "arithmetic_error": arithmetic_error,
        "qty_error": qty_error,
        "confidence": max(0.85, 1.0 - arithmetic_error),
    }


def bill_row_description(row, words, boxes):
    """Remove PO and the four numeric bill columns; keep material description."""
    nums = row_numbers(row, words, boxes)
    if len(nums) < 4:
        return ""

    numeric_indices = {n["index"] for n in nums[-4:]}
    desc = []

    for idx in row["indices"]:
        token = clean_text(words[idx])
        if not token or idx in numeric_indices:
            continue

        compact = re.sub(r"[^A-Za-z0-9]", "", token)
        if compact.isdigit() and len(compact) >= 6:
            # PO/document/material identifiers do not belong in description.
            continue

        desc.append(token)

    return clean_text(" ".join(desc))


def reconstruct_line_items(words, boxes):
    rows = candidate_rows(words, boxes)
    header_idx, end_idx = find_bill_table_bounds(rows)

    if header_idx is None:
        return []

    items = []

    for row in rows[header_idx + 1:end_idx]:
        text = clean_text(row["text"])
        upper = _norm_upper(text)

        if not text or is_consumption_row(text) or is_summary_row(text):
            continue

        if re.search(r"\bTOTAL\b", upper):
            continue

        columns = bill_row_numeric_columns(row, words, boxes)
        if columns is None:
            continue

        description = bill_row_description(row, words, boxes)
        if len(description) < 2:
            continue

        if re.fullmatch(r"[\d\s.,/_-]+", description):
            continue

        items.append({
            "description": description,
            "quantity": columns["quantity"],
            "unit_price": columns["unit_price"],
            "amount": columns["amount"],
            "confidence": round(float(columns["confidence"]), 4),
            "_y": row["y_center"],
        })

    # Preserve distinct invoice rows. Only exact duplicates are removed.
    deduped = []
    seen = set()
    for item in items:
        sig = (
            re.sub(r"\W+", "", item["description"].upper()),
            item["quantity"],
            item["unit_price"],
            item["amount"],
        )
        if sig in seen:
            continue
        seen.add(sig)
        item.pop("_y", None)
        deduped.append(item)

    return deduped


def extract_customer_from_invoice_header(rows):
    """
    Prefer the buyer/customer printed in the invoice header.
    For this invoice the header is:
      Form ZTPMKS259 SRI BAGAWATHIAMMAN TEX BILL NO
    """
    for row in rows[:20]:
        t = clean_text(row["text"])
        u = _norm_upper(t)

        if "BILL NO" not in u:
            continue

        left = re.split(r"\bBILL\s*NO\b", t, flags=re.I)[0]
        left = re.sub(r"\bFORM\s+[A-Z0-9_-]+\b", "", left, flags=re.I)
        left = re.sub(r"\bREF\s*NO\b.*$", "", left, flags=re.I)
        left = clean_text(left)

        # Avoid returning the title itself.
        if left and len(left) >= 3:
            return left

    # Generic label-aware fallback.
    customer_labels = (
        "CUSTOMER NAME", "CUSTOMER", "BUYER", "BILL TO", "SOLD TO",
        "CONSIGNEE", "SHIP TO",
    )
    for row in rows[:30]:
        t = clean_text(row["text"])
        u = _norm_upper(t)
        for label in customer_labels:
            pos = u.find(label)
            if pos >= 0:
                value = clean_text(t[pos + len(label):].lstrip(" :#-"))
                if value and len(value) > 2:
                    return value

    return None


def merge_fields(model_spans, words, boxes):
    fields = {}
    metadata = {}

    # Model candidates remain useful, but are not allowed to override stronger
    # structural/header evidence below.
    for span in model_spans:
        field = canonical_field(span.get("label"))
        if field not in CANONICAL_FIELDS:
            continue

        value = clean_text(span.get("value"))
        if not value:
            continue

        candidate = {
            "value": value,
            "source": "layoutlm",
            "confidence": float(span.get("confidence", 0)),
            "raw": value,
            "bbox": span.get("box"),
        }

        old = metadata.get(field)
        if old is None or candidate["confidence"] > old["confidence"]:
            fields[field] = value
            metadata[field] = candidate

    rows = candidate_rows(words, boxes)

    # --------------------------------------------------------
    # CUSTOMER — structural header evidence beats arbitrary model span.
    # --------------------------------------------------------
    customer = extract_customer_from_invoice_header(rows)
    if customer:
        fields["CUSTOMER_NAME"] = customer
        metadata["CUSTOMER_NAME"] = {
            "source": "ocr_invoice_header",
            "confidence": 1.0,
            "raw": customer,
        }

    # --------------------------------------------------------
    # INVOICE NUMBER
    # --------------------------------------------------------
    inv = extract_invoice_number(rows)
    if inv:
        fields["INVOICE_NUMBER"] = inv["value"]
        metadata["INVOICE_NUMBER"] = {
            "source": "ocr_labeled_invoice_number",
            "confidence": 1.0,
            "raw": inv["raw"],
            "bbox": inv["box"],
        }

    # --------------------------------------------------------
    # REF NO
    # --------------------------------------------------------
    # OCR may place the reference value on the same line or the next line.
    ref_value = None
    for pos, row in enumerate(rows[:30]):
        upper = _norm_upper(row["text"])
        if "REF NO" not in upper and "REFERENCE NO" not in upper:
            continue

        tail = re.sub(r".*?REF(?:ERENCE)?\s*NO\.?", "", row["text"], flags=re.I)
        tokens = re.findall(r"[A-Za-z0-9][A-Za-z0-9./_-]{0,15}", tail)
        for token in tokens:
            if DATE_RE.fullmatch(token):
                continue
            digits = re.sub(r"\D", "", token)
            if (digits and len(digits) <= 6) or (not digits and len(token) <= 6):
                ref_value = token
                break
        if ref_value:
            break

        for nxt in rows[pos + 1:pos + 4]:
            candidate = clean_text(nxt["text"]).lstrip(" :#-/.")
            tokens = re.findall(r"[A-Za-z0-9][A-Za-z0-9./_-]{0,15}", candidate)
            for token in tokens:
                if DATE_RE.fullmatch(token):
                    continue
                digits = re.sub(r"\D", "", token)
                if (digits and len(digits) <= 6) or (not digits and len(token) <= 6):
                    ref_value = token
                    break
            if ref_value:
                break
        if ref_value:
            break

    if ref_value:
        fields["REF_NO"] = ref_value
        metadata["REF_NO"] = {
            "source": "ocr_labeled_ref_no",
            "confidence": 1.0,
            "raw": ref_value,
        }

    # --------------------------------------------------------
    # DATES
    # --------------------------------------------------------
    invoice_date_candidate = extract_labeled_value(
        rows,
        ["INVOICE DATE", "BILL DATE"],
        exclude_labels=["DUE DATE", "DELIVERY DATE"],
    )

    # Only an explicit due-date label may populate DUE_DATE.
    due_date_candidate = extract_labeled_value(
        rows,
        ["DUE DATE", "PAYMENT DUE", "DUE ON"],
    )

    # The current invoice has two bare DATE labels. The first is the invoice
    # date. Do not call the second one a due date without an explicit label.
    if invoice_date_candidate is None:
        date_values = []
        for row in rows[:30]:
            for d in DATE_RE.findall(row["text"]):
                normalized = normalize_date(d)
                if normalized:
                    date_values.append((normalized, row["text"]))

        if date_values:
            fields["INVOICE_DATE"] = date_values[0][0]
            metadata["INVOICE_DATE"] = {
                "source": "ocr_first_header_date",
                "confidence": 0.98,
                "raw": date_values[0][1],
            }
            if len(date_values) > 1:
                metadata["_SECOND_HEADER_DATE"] = {
                    "value": date_values[1][0],
                    "raw": date_values[1][1],
                    "source": "ocr_second_header_date_unclassified",
                }

    if due_date_candidate:
        normalized = normalize_date(due_date_candidate["value"])
        if normalized:
            fields["DUE_DATE"] = normalized
            metadata["DUE_DATE"] = {
                "source": "ocr_labeled_due_date",
                "confidence": 1.0,
                "raw": due_date_candidate["value"],
                "row": due_date_candidate["raw_row"],
            }

    # Normalize any model-provided due date before validation. Never swap
    # dates; if it is earlier than the invoice date, reject it.
    if fields.get("DUE_DATE"):
        normalized_model_due = normalize_date(fields["DUE_DATE"])
        if normalized_model_due:
            fields["DUE_DATE"] = normalized_model_due
        else:
            fields.pop("DUE_DATE", None)
            metadata.pop("DUE_DATE", None)

    if fields.get("INVOICE_DATE") and fields.get("DUE_DATE"):
        try:
            inv_d = dt.datetime.strptime(fields["INVOICE_DATE"], "%Y-%m-%d").date()
            due_d = dt.datetime.strptime(fields["DUE_DATE"], "%Y-%m-%d").date()
            if due_d < inv_d:
                fields.pop("DUE_DATE", None)
                metadata.pop("DUE_DATE", None)
        except Exception:
            pass

    # --------------------------------------------------------
    # FINANCIAL ROWS
    # --------------------------------------------------------
    money, money_diag = extract_money_fields(words, boxes)
    for field, candidate in money.items():
        if candidate.get("value") is None:
            continue
        fields[field] = candidate["value"]
        metadata[field] = {
            "source": "ocr_labeled_financial_row",
            "confidence": 1.0,
            "raw": candidate.get("raw"),
            "keyword": candidate.get("keyword"),
            "row_text": candidate.get("row_text"),
        }

    # --------------------------------------------------------
    # GSTIN — only valid GSTIN structure is accepted.
    # --------------------------------------------------------
    gstins = extract_gstins(words, boxes)
    if gstins:
        fields["GSTIN"] = gstins[0]["value"]
        metadata["GSTIN"] = {
            "source": "ocr_gstin_regex",
            "confidence": 1.0,
            "raw": gstins[0]["raw"],
        }

    for field in MONEY_FIELDS:
        if fields.get(field) is not None:
            fields[field] = normalize_money(fields[field])

    return fields, metadata, money_diag, gstins


In [ ]:
# ============================================================
# CELL 8 — PAGE PIPELINE + RECONCILIATION + MULTI-PAGE MERGE
# ============================================================

def run_page_pipeline(image, words, boxes, page_number):
    started = time.perf_counter()

    if not words:
        raise RuntimeError(f"Page {page_number}: OCR returned no words.")

    if len(words) != len(boxes):
        raise RuntimeError(
            f"Page {page_number}: OCR word/box mismatch "
            f"({len(words)} vs {len(boxes)})"
        )

    print("OCR words:", len(words))
    print("Running Model A...")

    preds_a = run_token_classifier(
        image, words, boxes,
        model_a, model_a_processor, model_a_id2label,
    )

    print("Running Model B...")

    preds_b = run_token_classifier(
        image, words, boxes,
        model_b, model_b_processor, model_b_id2label,
    )

    spans_a = build_spans(preds_a)
    spans_b = build_spans(preds_b)

    print("Model A spans:", len(spans_a))
    print("Model B spans:", len(spans_b))

    fields, metadata, money_diag, gstins = merge_fields(
        spans_a + spans_b,
        words,
        boxes,
    )

    line_items = reconstruct_line_items(words, boxes)

    for item in line_items:
        item["page"] = page_number

    # Structural line-item diagnostics. The invoice-table detector is
    # deliberately independent of arithmetic selection.
    table_rows = candidate_rows(words, boxes)
    bill_header_idx, bill_end_idx = find_bill_table_bounds(table_rows)
    bill_table_detected = bill_header_idx is not None
    line_amount_sum = round(
        sum(money_float(i.get("amount")) or 0.0 for i in line_items), 2
    )
    extracted_subtotal = money_float(fields.get("SUBTOTAL"))
    line_sum_matches_subtotal = (
        abs(line_amount_sum - extracted_subtotal) <= 0.05
    ) if (line_items and extracted_subtotal is not None) else None

    elapsed = time.perf_counter() - started

    return {
        "fields": fields,
        "metadata": metadata,
        "line_items": line_items,
        "gstins": gstins,
        "diagnostics": {
            "page": page_number,
            "ocr_word_count": len(words),
            "model_a_predictions": len(preds_a),
            "model_b_predictions": len(preds_b),
            "model_a_spans": len(spans_a),
            "model_b_spans": len(spans_b),
            "inference_seconds": round(elapsed, 3),
            "money_diagnostics": {
                k: v[:10] for k, v in money_diag.items()
            },
            "bill_table_detected": bill_table_detected,
            "bill_table_header_row": bill_header_idx,
            "bill_table_end_row": bill_end_idx,
            "line_item_count": len(line_items),
            "line_item_amount_sum": f"{line_amount_sum:.2f}",
            "line_sum_matches_subtotal": line_sum_matches_subtotal,
        },
    }

def merge_page_results(page_results):
    final_fields = {}
    final_metadata = {}
    all_items = []
    page_diagnostics = []

    for page_result in page_results:
        page_diagnostics.append(page_result["diagnostics"])
        all_items.extend(page_result["line_items"])

        for field, value in page_result["fields"].items():
            if value is None or value == "":
                continue

            candidate = page_result["metadata"].get(field, {
                "source": "page",
                "confidence": 0.5,
                "raw": value,
            })

            old = final_metadata.get(field)

            # Explicit OCR financial/labeled evidence wins over generic model
            # spans. Otherwise confidence decides.
            explicit = str(candidate.get("source", "")).startswith("ocr_")
            old_explicit = (
                old is not None
                and str(old.get("source", "")).startswith("ocr_")
            )

            if (
                old is None
                or (explicit and not old_explicit)
                or candidate.get("confidence", 0) >
                   old.get("confidence", 0)
            ):
                final_fields[field] = value
                final_metadata[field] = candidate

    # Deduplicate cross-page line items.
    deduped = []

    for item in all_items:
        sig = (
            re.sub(r"\W+", "", item.get("description", "").upper()),
            item.get("quantity", ""),
            item.get("unit_price", ""),
            item.get("amount", ""),
        )

        if not any(
            (
                sig[0] == re.sub(
                    r"\W+", "",
                    old.get("description", "").upper()
                )
                and sig[1:] == (
                    old.get("quantity", ""),
                    old.get("unit_price", ""),
                    old.get("amount", ""),
                )
            )
            for old in deduped
        ):
            deduped.append(item)

    # Reconcile total tax if explicit tax rows exist.
    cgst = money_float(final_fields.get("CENTRAL_GST"))
    sgst = money_float(final_fields.get("STATE_GST"))

    if cgst is not None and sgst is not None:
        final_fields["TAX"] = f"{cgst + sgst:.2f}"
        final_metadata["TAX"] = {
            "source": "derived_cgst_plus_sgst",
            "confidence": 1.0,
            "raw": f"{cgst:.2f} + {sgst:.2f}",
        }

    warnings = []

    subtotal = money_float(final_fields.get("SUBTOTAL"))
    tax = money_float(final_fields.get("TAX"))
    total = money_float(final_fields.get("TOTAL_AMOUNT"))

    if subtotal is not None and tax is not None and total is not None:
        difference = abs((subtotal + tax) - total)

        if difference > 1.0:
            warnings.append(
                "Subtotal + tax does not reconcile with total "
                f"(difference={difference:.2f})."
            )

    if not deduped:
        warnings.append(
            "No confident invoice line items reconstructed from the BILL/INVOICE table."
        )

    # Document-level reconciliation. The bill table and BASIC TOTAL may be
    # on different pages, so page-local comparisons are not valid.
    global_line_amount_sum = round(
        sum(money_float(i.get("amount")) or 0.0 for i in deduped), 2
    )
    global_line_sum_matches_subtotal = None

    if subtotal is not None and deduped:
        global_line_sum_matches_subtotal = (
            abs(global_line_amount_sum - subtotal) <= 0.05
        )
        if not global_line_sum_matches_subtotal:
            warnings.append(
                "Invoice line-item amounts do not reconcile with SUBTOTAL; manual verification required."
            )

    for d in page_diagnostics:
        if d.get("bill_table_detected"):
            d["line_sum_matches_subtotal"] = global_line_sum_matches_subtotal

    return {
        "fields": final_fields,
        "field_metadata": final_metadata,
        "line_items": deduped,
        "diagnostics": {
            "pages": len(page_results),
            "page_diagnostics": page_diagnostics,
            "warnings": warnings,
            "global_line_item_amount_sum": f"{global_line_amount_sum:.2f}",
            "global_line_sum_matches_subtotal": global_line_sum_matches_subtotal,
        },
    }

def canonical_json(result):
    fields = result.get("fields", {})

    output = {
        field: fields.get(field)
        for field in CANONICAL_FIELDS
    }

    output["GSTIN"] = fields.get("GSTIN")
    output["line_items"] = result.get("line_items", [])
    output["_diagnostics"] = result.get("diagnostics", {})

    return output

def final_consistency_check(result):
    errors = []

    for field in CANONICAL_FIELDS + ["GSTIN", "line_items", "_diagnostics"]:
        if field not in result:
            errors.append(f"Missing top-level field: {field}")

    for field in ["INVOICE_DATE", "DUE_DATE"]:
        value = result.get(field)
        if value is not None and not re.fullmatch(
            r"\d{4}-\d{2}-\d{2}", str(value)
        ):
            errors.append(f"{field} is not normalized: {value}")

    for field in MONEY_FIELDS:
        value = result.get(field)
        if value is not None and not re.fullmatch(
            r"-?\d+\.\d{2}", str(value)
        ):
            errors.append(f"{field} is not decimal money: {value}")

    gstin = result.get("GSTIN")
    if gstin:
        normalized = re.sub(r"[^A-Za-z0-9]", "", str(gstin)).upper()
        if not (
            GSTIN_RE.fullmatch(normalized)
            and normalized[:2] in GST_STATE_CODES
        ):
            errors.append(f"Invalid GSTIN: {gstin}")

    seen = set()

    for i, item in enumerate(result.get("line_items", []), start=1):
        for key in [
            "description", "quantity", "unit_price",
            "amount", "confidence", "page"
        ]:
            if key not in item:
                errors.append(f"Line item {i} missing key {key}")

        for key in ["quantity", "unit_price", "amount"]:
            value = item.get(key)
            if value and looks_like_reference_number(value):
                errors.append(
                    f"Reference/document number leaked into "
                    f"line-item {key}: {value}"
                )

        sig = (
            item.get("description", "").upper().strip(),
            item.get("quantity", ""),
            item.get("unit_price", ""),
            item.get("amount", ""),
        )

        if sig in seen:
            errors.append(f"Duplicate line item detected: {i}")
        seen.add(sig)

    diagnostics = result.get("_diagnostics", {})
    if diagnostics.get("global_line_sum_matches_subtotal") is False:
        errors.append("Global invoice line-item amounts do not reconcile with SUBTOTAL.")

    inv = result.get("INVOICE_DATE")
    due = result.get("DUE_DATE")

    if inv and due:
        try:
            if dt.datetime.strptime(
                due, "%Y-%m-%d"
            ).date() < dt.datetime.strptime(
                inv, "%Y-%m-%d"
            ).date():
                errors.append("Due date is earlier than invoice date.")
        except Exception:
            pass

    return len(errors) == 0, errors


In [ ]:
# ============================================================
# CELL 9 — PDF/IMAGE PAGE LOADING
# ============================================================

def load_pages(input_path, pdf_dpi=170):
    input_path = Path(input_path)

    if input_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
        raise ValueError(
            f"Unsupported file type: {input_path.suffix}"
        )

    if input_path.suffix.lower() != ".pdf":
        image = Image.open(input_path).convert("RGB")
        return [image]

    pages = []
    doc = fitz.open(str(input_path))

    try:
        for page in doc:
            pix = page.get_pixmap(
                dpi=pdf_dpi,
                alpha=False,
            )

            image = Image.open(
                io.BytesIO(pix.tobytes("png"))
            ).convert("RGB")

            pages.append(image)

    finally:
        doc.close()

    return pages


In [ ]:
# ============================================================
# CELL 10 — ONE-CLICK INVOICE INFERENCE
# ============================================================

print("=" * 78)
print("UPLOAD ONE INVOICE")
print("=" * 78)
print("Supported: PDF / JPG / JPEG / PNG / WEBP / TIFF")
print("OCR      : PP-StructureV3")
print("Models   : LayoutLMv3 Model A + Model B")
print("Device   :", device)
print()

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No invoice was uploaded.")

uploaded_name = next(iter(uploaded))
uploaded_path = Path("/content") / uploaded_name

if uploaded_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
    raise ValueError(
        f"Unsupported upload type: {uploaded_path.suffix}"
    )

print("\nInput:", uploaded_path)
print("Size MB:", round(uploaded_path.stat().st_size / (1024 * 1024), 3))

page_images = load_pages(uploaded_path)

print("Pages:", len(page_images))

page_results = []
run_started = time.perf_counter()

for page_no, page_image in enumerate(page_images, start=1):

    print("\n" + "=" * 78)
    print(f"PAGE {page_no}/{len(page_images)}")
    print("=" * 78)

    page_file = TEMP_DIR / f"_page_{page_no}.png"

    try:
        page_image.save(page_file, format="PNG")

        words, boxes, approx, ocr_source = ocr_image_pp_structure(
            page_file
        )

        print("OCR source:", ocr_source)

        result = run_page_pipeline(
            page_image,
            words,
            boxes,
            page_no,
        )

        page_results.append(result)

        print("Page fields:", len(result["fields"]))
        print("Page line items:", len(result["line_items"]))

    except Exception as exc:
        print(
            f"ERROR — page {page_no} failed: {type(exc).__name__}: {exc}"
        )

        # Fail honestly. Do not generate a fake "complete" result.
        raise RuntimeError(
            f"Invoice extraction stopped on page {page_no}. "
            f"Original error: {type(exc).__name__}: {exc}"
        ) from exc

    finally:
        if page_file.exists():
            try:
                page_file.unlink()
            except Exception:
                pass

        try:
            page_image.close()
        except Exception:
            pass

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

if not page_results:
    raise RuntimeError("No page result was produced.")

merged = merge_page_results(page_results)
canonical_result = canonical_json(merged)

passed, consistency_errors = final_consistency_check(
    canonical_result
)

canonical_result["_diagnostics"]["consistency_check"] = {
    "status": "PASS" if passed else "WARNINGS",
    "errors": consistency_errors,
}

canonical_result["_diagnostics"]["source_file"] = uploaded_path.name
canonical_result["_diagnostics"]["device"] = device
canonical_result["_diagnostics"]["paddle_device"] = PADDLE_DEVICE
canonical_result["_diagnostics"]["model_a_source"] = str(model_a_source)
canonical_result["_diagnostics"]["model_b_source"] = str(model_b_source)
canonical_result["_diagnostics"]["total_seconds"] = round(
    time.perf_counter() - run_started,
    3,
)

print("\n" + "=" * 78)
print("GST INVOICE EXTRACTION COMPLETE")
print("=" * 78)


In [ ]:
# ============================================================
# CELL 11 — FINAL JSON + REPORT + SAVE
# ============================================================

safe_stem = re.sub(
    r"[^A-Za-z0-9._-]+",
    "_",
    uploaded_path.stem,
).strip("_") or "invoice"

json_path = OUTPUT_DIR / f"{safe_stem}_extracted_v11.json"
report_path = OUTPUT_DIR / f"{safe_stem}_report_v11.txt"

json_path.write_text(
    json.dumps(
        canonical_result,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

report = [
    "GST INVOICE EXTRACTION REPORT — v11",
    "=" * 78,
    f"Input: {uploaded_path.name}",
    f"Generated: {dt.datetime.now().isoformat(timespec='seconds')}",
    f"Device: {device}",
    f"Paddle device: {PADDLE_DEVICE}",
    "",
    "FIELDS",
    "-" * 78,
]

for field in CANONICAL_FIELDS + ["GSTIN"]:
    value = canonical_result.get(field)
    report.append(f"{field}: {value}")

report.extend([
    "",
    "LINE ITEMS",
    "-" * 78,
])

for i, item in enumerate(
    canonical_result.get("line_items", []),
    start=1,
):
    report.append(
        f"{i}. "
        f"description={item.get('description','')} | "
        f"quantity={item.get('quantity','')} | "
        f"unit_price={item.get('unit_price','')} | "
        f"amount={item.get('amount','')} | "
        f"confidence={item.get('confidence','')} | "
        f"page={item.get('page','')}"
    )

report.extend([
    "",
    "VALIDATION",
    "-" * 78,
    f"Status: {canonical_result['_diagnostics']['consistency_check']['status']}",
])

for error in canonical_result["_diagnostics"]["consistency_check"]["errors"]:
    report.append(f"- {error}")

report.extend([
    "",
    "WARNINGS",
    "-" * 78,
])

for warning in canonical_result["_diagnostics"].get("warnings", []):
    report.append(f"- {warning}")

report.extend([
    "",
    "RUNTIME",
    "-" * 78,
    f"Pages: {canonical_result['_diagnostics']['pages']}",
    f"Seconds: {canonical_result['_diagnostics']['total_seconds']}",
])

report_path.write_text(
    "\n".join(report),
    encoding="utf-8",
)

print("\nCANONICAL JSON")
print("=" * 78)
print(
    json.dumps(
        canonical_result,
        indent=2,
        ensure_ascii=False,
    )[:20000]
)

print("\nSaved JSON  :", json_path)
print("Saved report:", report_path)
print(
    "Consistency :",
    canonical_result["_diagnostics"]["consistency_check"]["status"],
)


## Run instructions

1. Select a **GPU runtime** in Colab if available.
2. Start from a clean Colab runtime when possible. If Cell 0 detects a native-package mismatch, it uses `os.kill()` and the runtime will reconnect; do not manually reinstall packages after that.
3. Run **all cells from top to bottom**.
4. Cell 0 installs/aligns dependencies.
5. Cell 1 validates `fitz`, `paddleocr`, PaddlePaddle, Transformers and GPU/CPU.
6. Cell 3 initializes PP-StructureV3 **before invoice upload**.
7. Cell 10 is the only upload/inference cell.
8. Do not manually install `pymupdf`, `paddleocr`, or PaddleOCR again between cells.

### Important

This notebook removes the specific runtime failures encountered in the previous runs, including `No module named 'fitz'`, `No module named 'paddleocr'`, and the CPU oneDNN/PIR error path. It does **not** mathematically guarantee zero errors for every future Colab image/package release; if Colab changes its base environment, Cell 0 will detect and align the required packages before inference.
